# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://773bf09f-c563-40e7-93ae-bdef148a111e.us-east-1-1.aws.cloud.qdrant.io:6333


## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [2]:
from langchain_core.documents import Document
import fitz

# TODO: PDF 파일 경로를 입력하세요
# 예시: "../datasets/your_document.pdf"
file_path = "../datasets/ISMS_P.pdf"

doc = fitz.open(file_path)
docs = []

# 페이지 단위로 Document 생성 (Parent Document)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": file_path.split("/")[-1],
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

총 263개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 215자
평균 페이지 길이: 1819자

첫 페이지 내용 미리보기:
  발 간 등 록 번 호
11-17903650-100016-14


정보보호 및
개인정보보호
관리체계(ISMS-P)
인증기준 안내서





  2023. 11.





             개인정보보호위원회

                                                     personal Information Protection Commission...


## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 263
  - Child chunk 수: 1855
  - 평균 chunk/page: 7.1

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: page_1
  Page: 1
  Length: 213자
  Content: 발 간 등 록 번 호
11-17903650-100016-14


정보보호 및
개인정보보호
관리체계(ISMS-P)
인증기준 안내서





  2023. 11.





      ...

Chunk 2:
  Parent ID: page_2
  Page: 2
  Length: 247자
  Content: 발 간 등 록 번 호
11-17903650-100016-14



   정보보호 및 개인정보보호

 관리체계(ISMS-P) 인증기준 안내서




                  ...

Chunk 3:
  Parent ID: page_3
  Page: 3
  Length: 369자
  Content: 정보보호 및 개인정보보호
 관리체계(ISMS-P) 인증기준 안내서





 안내
 사항




발간 목적

본 안내서는 「정보보호 및 개인정보보호 관리체계 인증 등에 관한 고시」...


## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [4]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://773bf09f-c563-40e7-93ae-bdef148a111e.us-east-1-1.aws.cloud.qdrant.io:6333


In [5]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "isms_p"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 'isms_p' 생성 완료

1855개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [6]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 263개의 Parent 문서 저장 완료

Docstore 키 예시: ['page_1', 'page_2', 'page_3', 'page_4', 'page_5']


## 5. Parent Document Retriever 구현

In [7]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

✓ Parent Document Retriever 생성 완료


## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [10]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "isms-p 자격증을 위한 가이드"

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: isms-p 자격증을 위한 가이드


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 4
  Parent ID: page_4
  길이: 115자
  내용: 이 안내서는 ISMS­P 홈페이지(https://isms-p.kisa.or.kr)에 게재되어 있으며, 기타 문의사항이 있으면
아래로 문의해 주시기 바랍니다.

E-mail : isms-p@kisa.or.kr

Chunk 2:
  페이지: 264
  Parent ID: page_264
  길이: 121자
  내용: 정보보호 및 개인정보보호 관리체계(ISMS-P)
      인증기준 안내서





    개인정보보호위원회

                 personal Information Protection Commission


[2] Parent Document 검색 결과
--------------------------------------------------------------------------------

Page 1:
  페이지 번호: 4
  Parent ID: page_4
  길이: 512자
  내용 미리보기: 이 안내서는 ｢정보통신망 이용촉진 및 정보보호 등에 관한 법률｣(이하 ʻ정보통신망법ʼ이라 함) 제47조(정보보호
관리체계의 인증) 및 ｢개인정보 보호법｣ 제32조의2(개인정보 보호 인증), ｢정보보호 및 개인정보보호 관리체계
인증 등에 관한 고시｣(과학기술정보통신부고시 제2023-33호, 개인정보보호위원회고시 제2023-8호)에서
규정하고 있는 사항에 대하여 보다 구체적으로 설명함을 목적으로 합니다.

이 안내서는 지속적인 보완 작업을 통하여 변경될 수 있으므로 항상 최신 버전 여부를 확인한 후 사용하여야 합니다.
또한 안내서에 제...

Page 2:
  페이지 번호: 264
  Parent ID: pag

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [13]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

# TODO: 시스템 프롬프트를 팀 문서 도메인에 맞게 수정하세요
# 예시: "당신은 의료 전문가입니다.", "당신은 법률 전문가입니다." 등
template = """
당신은 ISMS-P 전문가입니다.
주어진 정보를 바탕으로 사용자가 ISMS-P 자격증을 취득하고자 하는 사람들을 위해 설명하세요.
비전공자 기준으로 누구는 이해하기 쉽게 설명해야합니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.
또한, 답변에 참고한 문서의 출처와 페이지 번호를 명시하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content

print("✓ RAG 시스템 준비 완료")

✓ RAG 시스템 준비 완료


## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [14]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "ISMS-P 의 챕터별 내용을 간단하게 설명해줘.",
    "ISMS-P를 취득하기 위해 자격증 공부하는 사람들을 위한 객관식 문제를 5개를 만들어줘.",
    "ISMS-P 흐름을 알기 쉽게 설명해줘."
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: ISMS-P 의 챕터별 내용을 간단하게 설명해줘.



물론입니다.  
ISMS-P는 **정보보호(ISMS)**와 **개인정보보호(P)**를 함께 관리하는 인증제도이며, 크게 **3개 챕터**로 구성됩니다. 비전공자 기준으로 쉽게 말하면, **“회사나 기관이 정보와 개인정보를 안전하게 관리하고 있는지 점검하는 기준”**이라고 이해하시면 됩니다.

---

## ISMS-P 챕터별 내용 간단 설명

### 1장. ISMS-P 인증기준 개요
- ISMS-P가 **무엇인지**, 왜 필요한지, 전체 기준이 **어떻게 구성되어 있는지**를 설명하는 부분입니다.
- 쉽게 말해 **책의 목차와 전체 개념 소개** 같은 장입니다.

**출처:** ISMS_P.pdf, p.6 / p.5

---

### 2장. 정보보호 및 개인정보보호 관리체계(ISMS-P) 인증기준 설명
이 장은 실제 인증을 받기 위해 **무엇을 해야 하는지**를 자세히 설명합니다.  
크게 3가지 영역으로 나뉩니다.

#### 2-1. 관리체계 수립 및 운영
- 조직이 정보보호와 개인정보보호를 **체계적으로 운영할 수 있는 구조**를 만드는 부분입니다.
- 예: 정책 만들기, 책임자 정하기, 위험 찾기, 점검하고 개선하기

쉽게 말하면, **“보안 업무를 누가, 어떻게, 어떤 기준으로 운영할지 정하는 단계”**입니다.

#### 2-2. 보호대책 요구사항
- 실제로 정보를 안전하게 지키기 위한 **기술적·관리적·물리적 보안조치**를 다룹니다.
- 예:
  - 직원 보안교육
  - 출입통제
  - 비밀번호/권한 관리
  - 암호화
  - 개발 시 보안 적용
  - 사고 대응
  - 재해복구

쉽게 말하면, **“정보를 훔치거나 잃어버리지 않도록 실제 보호장치를 마련하는 단계”**입니다.

#### 2-3. 개인정보 처리 단계별 요구사항
- 개인정보를 **언제, 어떻게 다루는지** 단계별로 보호조치를 설명합니다.
- 단계는 다음과 같습니다.
  - 수집할 때
  - 보유·이용할 때
  - 제공할 때
  - 파기할 때
  - 정보주체 권리를 보호할 때

쉽게 말하면, **“개인정보를 처음 받는 순간부터 없애는 순간까지 전 과정에서 안전하게 다루는 방법”**입니다.

**출처:** ISMS_P.pdf, p.5

---

## 한눈에 정리
- **1장:** ISMS-P가 무엇인지 전체 개요
- **2장:** 인증을 받기 위한 실제 기준 설명
  - **관리체계 수립 및 운영**: 운영 구조 만들기
  - **보호대책 요구사항**: 보안조치 마련하기
  - **개인정보 처리 단계별 요구사항**: 개인정보 전 과정 보호하기

---

원하시면 다음 답변에서는 제가 이어서  
**“ISMS-P 시험 준비할 때 챕터별로 어떻게 공부해야 하는지”**도 비전공자 기준으로 쉽게 정리해드릴게요.


질문: ISMS-P를 취득하기 위해 자격증 공부하는 사람들을 위한 객관식 문제를 5개를 만들어줘.



물론입니다.  
비전공자도 이해하기 쉽게, **ISMS-P 인증기준**을 바탕으로 한 **객관식 문제 5개**를 만들어드리겠습니다.

ISMS-P는 크게  
1) **관리체계 수립 및 운영**,  
2) **보호대책 요구사항**,  
3) **개인정보 처리 단계별 요구사항**  
으로 구성됩니다. 그중 이번 문제는 특히 **보호대책 요구사항**과 **기본 구성**을 이해하는 데 도움이 되도록 만들었습니다.  
(출처: **ISMS_P.pdf, p.8, p.10**)

---

## 객관식 문제 5개

### 1. ISMS-P 인증기준은 크게 몇 개의 영역으로 구성되어 있는가?
① 2개  
② 3개  
③ 4개  
④ 5개  

**정답: ② 3개**  
**해설:** ISMS-P는 **관리체계 수립 및 운영, 보호대책 요구사항, 개인정보 처리 단계별 요구사항**의 3개 영역으로 구성됩니다.  
**출처:** ISMS_P.pdf, **p.8**

---

### 2. 다음 중 ISMS-P에서만 추가로 요구되는 영역은 무엇인가?
① 관리체계 수립 및 운영  
② 보호대책 요구사항  
③ 개인정보 처리 단계별 요구사항  
④ 위험 관리  

**정답: ③ 개인정보 처리 단계별 요구사항**  
**해설:** ISMS와 ISMS-P는 모두 **1번, 2번 영역**을 포함하지만, **3번 개인정보 처리 단계별 요구사항**은 **ISMS-P에만** 있습니다.  
**출처:** ISMS_P.pdf, **p.8**

---

### 3. 보호대책 요구사항 영역은 총 몇 개의 분야로 구성되어 있는가?
① 10개  
② 11개  
③ 12개  
④ 16개  

**정답: ③ 12개**  
**해설:** 보호대책 요구사항은 **12개 분야, 64개 인증기준**으로 구성되어 있습니다.  
**출처:** ISMS_P.pdf, **p.10**

---

### 4. 다음 중 보호대책 요구사항의 분야에 해당하는 것은?
① 개인정보 제공 시 보호조치  
② 인적 보안  
③ 정보주체 권리보호  
④ 개인정보 파기 시 보호조치  

**정답: ② 인적 보안**  
**해설:** **인적 보안**은 보호대책 요구사항의 분야입니다. 반면 ①, ③, ④는 **개인정보 처리 단계별 요구사항**에 해당합니다.  
**출처:** ISMS_P.pdf, **p.10, p.8**

---

### 5. 다음 중 보호대책 요구사항의 세부 항목으로 맞는 것은?
① 정책의 유지관리  
② 개인정보 수집 시 보호조치  
③ 정보주체 권리보호  
④ 개인정보 파기 시 보호조치  

**정답: ① 정책의 유지관리**  
**해설:** **정책의 유지관리**는 보호대책 요구사항의 세부 항목(2.1.1)입니다.  
②, ③, ④는 개인정보 처리 단계별 요구사항에 속합니다.  
**출처:** ISMS_P.pdf, **p.10, p.8**

---

## 핵심 정리
- **ISMS-P는 3개 영역**으로 구성됩니다.  
- 그중 **보호대책 요구사항**은 **12개 분야, 64개 인증기준**입니다.  
- 비전공자라면 먼저 **“ISMS-P 전체 구조”**와 **“어느 항목이 어느 영역에 속하는지”**를 구분해서 공부하는 것이 좋습니다.

원하시면 다음 단계로  
**난이도별 객관식 20문항**,  
**정답/오답 해설 포함 버전**,  
또는 **기출 스타일 문제**로도 만들어드릴게요.


질문: ISMS-P 흐름을 알기 쉽게 설명해줘.



ISMS-P에서 말하는 **“흐름”**은 한마디로,  
**개인정보가 언제, 어디서, 왜, 어떻게 들어와서 수집되고, 보관·이용되다가, 최종적으로 어떻게 파기되는지**를 한눈에 정리한 것입니다.

비전공자 기준으로 쉽게 말하면,  
개인정보가 **“들어오는 길 → 사용되는 길 → 사라지는 길”**을 지도처럼 그려두는 거예요.

---

## 1) 왜 흐름을 정리해야 하나요?
ISMS-P는 조직이 개인정보를 **대충 다루지 않고, 정확히 관리하고 있는지** 확인합니다.  
그래서 “우리 회사에서 개인정보가 실제로 어떻게 움직이는지”를 문서로 남겨야 합니다.

즉, 다음을 파악하는 것이 핵심입니다.

- 어떤 서비스에서 개인정보가 수집되는지
- 어떤 업무에서 개인정보를 사용하는지
- 어디에 저장되는지
- 누구에게 전달되는지
- 언제/어떻게 삭제되는지

이걸 정리해 두면, 개인정보가 새로 생기거나 업무가 바뀌어도 놓치지 않고 관리할 수 있습니다.

---

## 2) ISMS-P 흐름은 어떻게 잡나요?
참고 문서에서는 다음 2단계로 설명합니다.

### 1단계: 개인정보가 처리되는 단위 업무를 찾기
예를 들어,
- 회원가입
- 주문 처리
- 고객상담
- 배송
- 탈퇴 처리

이런 식으로 **개인정보가 들어가는 업무 하나하나를 찾는 것**입니다.

### 2단계: 개인정보 생명주기별로 흐름표 작성
개인정보는 보통 아래 순서로 흐릅니다.

1. **수집**  
   - 고객이 회원가입할 때 이름, 연락처를 입력
2. **보유·이용**  
   - 주문 처리, 고객 응대, 마케팅 동의 관리 등에 사용
3. **제공/위탁/연계**  
   - 배송업체나 결제대행사(PG)에 일부 정보 전달 가능
4. **파기**  
   - 탈퇴, 보관기간 만료 시 안전하게 삭제

즉, 개인정보의 **전 생애주기**를 기준으로 흐름을 정리하는 것입니다.

---

## 3) 왜 자주 업데이트해야 하나요?
개인정보 흐름은 고정되어 있지 않습니다.  
서비스가 바뀌거나 조직이 바뀌면 흐름도 바뀝니다.

예를 들면:
- 신규 서비스 오픈
- 업무 절차 변경
- 개인정보 처리 방식 변경
- 조직 개편
- 외부 시스템 연계 추가
- 법규 개정
- 새로운 취약점 발생

그래서 **최소 연 1회 이상** 주기적으로 검토하고 최신 상태로 유지해야 합니다.

---

## 4) 문서로는 무엇을 남기나요?
참고 문서에 따르면 증거자료 예시는 다음과 같습니다.

- 정보서비스 현황표
- 정보서비스 업무흐름표/업무흐름도
- 개인정보 처리 현황표
- 개인정보 흐름표/흐름도

쉽게 말해,
**“어떤 서비스가 있고, 그 서비스에서 개인정보가 어떤 경로로 움직이는지”**를 표와 그림으로 보여주는 자료입니다.

---

## 5) 자주 나오는 실수
ISMS-P에서 자주 지적되는 문제도 있습니다.

- 문서에 실제 업무 흐름이 반영되지 않은 경우
- 개인정보 흐름도와 실제 흐름이 다른 경우
- 중요한 흐름이 빠져 있는 경우
- 처음 만든 뒤 업데이트를 안 한 경우

즉, **문서만 만든다고 끝이 아니라, 실제와 일치해야** 합니다.

---

## 6) 아주 쉬운 예시
예를 들어 온라인 쇼핑몰이 있다고 해보겠습니다.

- 고객이 회원가입 → 이름, 휴대폰번호 수집
- 상품 주문 → 배송지 주소 수집
- 결제 → 결제정보 처리
- 배송 → 택배사에 배송지 정보 제공
- 탈퇴/보관기간 종료 → 개인정보 파기

이 전체 과정을 그림으로 그리면 그게 바로 **개인정보 흐름도**입니다.

---

## 한 줄 정리
ISMS-P의 개인정보 흐름은  
**“개인정보가 수집되어 사용되고, 전달되고, 마지막에 파기되는 과정 전체를 문서로 정리해 관리하는 것”**입니다.

---

## 참고한 출처
1. **ISMS_P.pdf, 페이지 33**  
   - 개인정보 처리 현황 식별, 개인정보 흐름 파악, 개인정보 흐름표/흐름도 문서화
   - 개인정보 단위 업무 식별 및 생명주기별 흐름표 작성

2. **ISMS_P.pdf, 페이지 36**  
   - 서비스/업무/정보자산/개인정보 흐름의 주기적 검토
   - 최신성 유지
   - 증거자료 예시 및 결함사례

원하시면 다음 단계로  
**“개인정보 흐름도 예시를 그림처럼 쉽게 설명”**해드릴게요.

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합